In [1]:
# Librerías
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import scipy.optimize as sco

from itertools import combinations
import os

# Paleta de colores BBVA
paleta_bbva = {
    "azul_logotipo": "#004481",
    "azul_oscuro_digital": "#072146",
    "azul_celeste_aqua": "#5BBEFF",
    "azul_medio": "#1973B8",
    "azul_claro": "#D4EDFC"
}

colores_bbva_list = list(paleta_bbva.values())

In [2]:
ruta_graficos = os.path.join('..', 'DATOS', 'GRAFICOS', 'CARETERA')
if not os.path.exists(ruta_graficos):
    os.makedirs(ruta_graficos) 

In [3]:
# Cargar df_diario.parquet desde la raíz del proyecto
ruta_archivo = os.path.join('..', 'DATOS', 'TRANSFORMADOS', 'df_diario.parquet')

df = pd.read_parquet(ruta_archivo)

df.head()

Ticker,AAPL,AMGN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,DOW,...,KO,MCD,MMM,MRK,MSFT,NKE,PG,TRV,UNH,V
Atributo,open,open,open,open,open,open,open,open,open,open,...,adj close,adj close,adj close,adj close,adj close,adj close,adj close,adj close,adj close,adj close
2018-06-06,45.803260,147.060853,90.831614,348.858528,131.109462,133.374320,35.413420,90.796595,96.121399,NaN,...,33.519554,133.685410,128.639160,45.619339,94.516289,66.114014,59.448196,107.609100,215.247849,128.590698
2018-06-07,45.927449,148.240513,92.139591,358.040544,134.061884,134.585641,35.801869,91.843793,97.898584,NaN,...,33.504139,139.530716,129.402710,45.663544,93.031555,66.122833,60.575584,109.097122,215.533508,126.288361
2018-06-08,45.176401,148.579864,92.003343,354.325406,133.276850,130.763026,35.219198,93.746452,98.071492,NaN,...,33.897491,139.061462,130.361816,46.098145,93.723206,66.246651,61.710964,110.400223,217.005142,127.137604
2018-06-11,45.262742,148.781859,91.903427,357.078069,132.355285,132.927524,35.300123,93.488335,99.839069,NaN,...,34.036320,137.069077,129.434250,45.914001,93.188316,65.972488,61.990837,109.710815,219.688751,126.354393
2018-06-12,45.272204,148.886901,92.194088,357.309054,133.703498,133.999842,35.348680,94.129931,100.607582,NaN,...,34.252258,137.266693,129.308075,45.995022,93.428085,65.707138,61.758961,108.937401,219.515549,127.250809


In [4]:
# Extraer y limpiar nombres de columnas
df_close = df[[c for c in df.columns if c[1] == "close"]].copy()
df_close.columns = [c[0] for c in df_close.columns]

# Graficar evolución de los precios de cierre
fig = px.line(
    df_close,
    title="Evolución del Precio de Cierre",
    labels={"value": "Precio", "index": "Fecha"},
    color_discrete_sequence=colores_bbva_list
)

fig.write_image(os.path.join(ruta_graficos, ('Evolución del Precio de Cierre' + '.png').replace(' ', '_')))

fig.show()

In [12]:
# 1. Preparar retornos
activos = ['AXP', 'CRM', 'MSFT', 'KO', 'JNJ', 'GS', 'PG', 'DIS', 'UNH', 'MRK', 'V', 'CVX', 'BA', 'INTC', 'MMM', 'IBM', 'CSCO', 'AAPL', 'MCD', 'CAT', 'HD', 'AMGN', 'JPM', 'NKE', 'TRV', 'DOW']
retornos = df.xs("close", axis=1, level=1).filter(items=activos).pct_change().dropna()

r_anual = retornos.mean() * 252
cov_anual = retornos.cov() * 252
matriz_corr = retornos.corr()

# 2. Calcular métricas para cada trío
resultados = []
for trio in combinations(retornos.columns, 3):
    t = list(trio)
    rent = r_anual[t].mean()
    vol = np.sqrt(cov_anual.loc[t, t].values.sum() / 9)
    corr_media = (matriz_corr.loc[t, t].values.sum() - 3) / 6
    
    resultados.append({
        "Trio": " - ".join(t), 
        "Corr": round(corr_media, 4), 
        "Sharpe": round(rent / vol, 4), 
        "Rent": rent, 
        "Vol": vol
    })

df_t = pd.DataFrame(resultados)

# 3. Identificar el mejor trío
top = df_t.loc[(df_t["Sharpe"] - df_t["Corr"]).idxmax()]
opt_label = f"ÓPTIMO: {top['Trio']}"
df_t["Cat"] = np.where(df_t.index == top.name, opt_label, "Otros")

# 4. Crear gráfico
fig = px.scatter(
    df_t, x="Corr", y="Sharpe", color="Cat", hover_data=["Trio", "Rent", "Vol"],
    color_discrete_map={"Otros": paleta_bbva["azul_celeste_aqua"], opt_label: paleta_bbva["azul_oscuro_digital"]},
    title="Sharpe vs Correlación",
    labels={"Corr": "Correlación Media", "Sharpe": "Ratio Sharpe"}
)

fig.add_annotation(
    x=top["Corr"], y=top["Sharpe"], 
    text=f"<b>{top['Trio']}</b><br>Sharpe: {top['Sharpe']:.2f} | Corr: {top['Corr']:.2f}",
    showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_logotipo"], ax=-80, ay=-30, font=dict(color=paleta_bbva["azul_logotipo"])
)

fig.update_traces(marker=dict(size=5, opacity=0.4), selector=dict(name="Otros"))
fig.update_traces(marker=dict(size=14, symbol="star"), selector=dict(name=opt_label))
fig.update_layout(height=700, width=850).show()
fig.write_image(os.path.join(ruta_graficos, ("Sharpe vs Correlación" + '.png').replace(' ', '_')))

In [6]:
cartera_tickers = ['MSFT', 'PG', 'CAT']
matriz_corr = retornos[cartera_tickers].corr().round(4)
print(matriz_corr)

        MSFT      PG     CAT
MSFT  1.0000  0.3560  0.2281
PG    0.3560  1.0000  0.0554
CAT   0.2281  0.0554  1.0000


In [14]:
# 1. Preparar datos de retornos
df_close = df.xs("close", axis=1, level=1).filter(items=activos)
df_close.index = pd.to_datetime(df[("date", "date")] if ("date", "date") in df.columns else df.get("date", df.index))
ret = df_close.pct_change().dropna()

r_d, cov_d, corr_m = ret.mean(), ret.cov(), ret.corr()
w = np.array([1/3, 1/3, 1/3])
resultados = []

# 2. Filtrar tríos con correlación asimétrica
for trio in combinations(ret.columns, 3):
    t = list(trio)
    c_sub = corr_m.loc[t, t].values
    r12, r13, r23 = c_sub[0, 1], c_sub[0, 2], c_sub[1, 2]
    _, r_min2, r_max = sorted([r12, r13, r23])
    
    if r_max >= 0.50 and r_min2 <= 0.25:
        rent = np.dot(r_d[t], w) * 252
        vol = np.sqrt(w.T @ cov_d.loc[t, t] @ w) * np.sqrt(252)
        
        if r_max == r12: bloque = f"{t[0]}-{t[1]} + {t[2]}"
        elif r_max == r13: bloque = f"{t[0]}-{t[2]} + {t[1]}"
        else: bloque = f"{t[1]}-{t[2]} + {t[0]}"
            
        resultados.append({
            "Trio": " - ".join(t), "Estructura": bloque,
            "Corr_Max_Par": round(r_max, 4), "Corr": round((r12 + r13 + r23) / 3, 4),
            "Sharpe": round(rent / vol, 4), "Rent": rent, "Vol": vol
        })

df_t = pd.DataFrame(resultados)

# 3. Identificar el óptimo y graficar
top = df_t.loc[df_t["Sharpe"].idxmax()]
opt_cat = f"ÓPTIMO ASIMÉTRICO: {top['Trio']}"
df_t["Cat"] = np.where(df_t.index == top.name, opt_cat, "Otros Asimétricos")

fig = px.scatter(
    df_t, x="Corr", y="Sharpe", color="Cat", hover_data=["Trio", "Estructura", "Corr_Max_Par", "Rent", "Vol"],
    color_discrete_map={"Otros Asimétricos": paleta_bbva["azul_celeste_aqua"], opt_cat: paleta_bbva["azul_oscuro_digital"]},
    title="Estrategia Asimétrica: Sharpe vs Correlación",
    labels={"Corr": "Correlación Media", "Sharpe": "Ratio Sharpe"}
)

fig.add_annotation(
    x=top["Corr"], y=top["Sharpe"], 
    text=f"<b>{top['Trio']}</b><br>{top['Estructura']}<br>Sharpe: {top['Sharpe']:.2f} | Corr Media: {top['Corr']:.2f}",
    showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_logotipo"], ax=-120, ay=-40, font=dict(color=paleta_bbva["azul_logotipo"])
)

fig.update_traces(marker=dict(size=5, opacity=0.4), selector=dict(name="Otros Asimétricos"))
fig.update_traces(marker=dict(size=14, symbol="star"), selector=dict(name=opt_cat))
fig.update_layout(height=700, width=850).show()
fig.write_image(os.path.join(ruta_graficos, ('Estrategia Asimétrica Sharpe vs Correlación' + '.png').replace(' ', '_')))

In [8]:
cartera_tickers = ['GS', 'PG', 'V']
matriz_corr = ret[cartera_tickers].corr().round(4)
print(matriz_corr)

        GS      PG       V
GS  1.0000  0.1008  0.5415
PG  0.1008  1.0000  0.2297
V   0.5415  0.2297  1.0000


In [15]:
# Frontera eficiente: tres activos y pesos de 5 %

distribuciones = np.array([
    [i, j, 20-i-j]
    for i in range(1, 19)
    for j in range(1, 20-i)
]) / 20

resultados = []

for trio in combinations(retornos.columns, 3):
    t = list(trio)
    pesos = distribuciones
    rent = pesos @ r_anual[t].values
    cov = cov_anual.loc[t, t].values
    vol = np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", pesos, cov, pesos), 0))

    resultados.extend([
        {"Trio": " - ".join(t), "Rent": r, "Vol": v, "Sharpe": r/v, "Pesos": w}
        for r, v, w in zip(rent, vol, pesos)
    ])

df_carteras = pd.DataFrame(resultados)
mejor_libre = df_carteras.loc[df_carteras["Sharpe"].idxmax()]

# Frontera eficiente
orden = df_carteras.sort_values(["Vol", "Rent"], ascending=[True, False])
df_frontera_libre = orden[
    orden["Rent"] > orden["Rent"].cummax().shift(fill_value=-np.inf) + 1e-8
]

# Etiqueta de la cartera óptima
activos_optimos = mejor_libre["Trio"].split(" - ")
composicion = "<br>".join(
    f"{a}: {p:.0%}" for a, p in zip(activos_optimos, mejor_libre["Pesos"])
)

etiqueta = (
    f"<b>CARTERA ÓPTIMA</b><br>{mejor_libre['Trio']}<br>"
    f"Rentabilidad: {mejor_libre['Rent']:.2%}<br>"
    f"Volatilidad: {mejor_libre['Vol']:.2%}<br>"
    f"Sharpe: {mejor_libre['Sharpe']:.3f}<br>{composicion}"
)

# Gráfico
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_carteras["Vol"], y=df_carteras["Rent"],
    mode="markers", name="Carteras",
    marker=dict(color=paleta_bbva["azul_celeste_aqua"], size=4, opacity=0.35)
))

fig.add_trace(go.Scatter(
    x=df_frontera_libre["Vol"], y=df_frontera_libre["Rent"],
    mode="lines", name="Frontera eficiente",
    line=dict(color=paleta_bbva["azul_logotipo"], width=2.5)
))

fig.add_trace(go.Scatter(
    x=[mejor_libre["Vol"]], y=[mejor_libre["Rent"]],
    mode="markers", name="Cartera óptima",
    marker=dict(color=paleta_bbva["azul_oscuro_digital"], size=17, symbol="star")
))

fig.add_annotation(
    x=mejor_libre["Vol"], y=mejor_libre["Rent"], text=etiqueta,
    showarrow=True, ax=100, ay=-90,
    bgcolor="white", bordercolor=paleta_bbva["azul_logotipo"],
    borderpad=7, font=dict(size=11)
)

fig.update_layout(
    title="Frontera Eficiente - Tres Activos sin Restricción de Pesos",
    xaxis_title="Volatilidad Anualizada",
    yaxis_title="Rentabilidad Anualizada",
    xaxis_tickformat=".0%", yaxis_tickformat=".0%",
    width=1100, height=750
)

fig.write_image(os.path.join(ruta_graficos, ('Frontera Eficiente - Tres Activos sin Restricción de Pesos' + '.png').replace(' ', '_')))

fig.show()

print("Cartera óptima:", mejor_libre["Trio"])
print(f"Rentabilidad: {mejor_libre['Rent']:.2%} | Volatilidad: {mejor_libre['Vol']:.2%}")
print(f"Sharpe: {mejor_libre['Sharpe']:.3f}")
print(composicion.replace("<br>", "\n"))
print(f"Total de carteras analizadas: {len(df_carteras):,}")

Cartera óptima: MSFT - GS - AAPL
Rentabilidad: 27.21% | Volatilidad: 24.60%
Sharpe: 1.106
MSFT: 35%
GS: 35%
AAPL: 30%
Total de carteras analizadas: 444,600


In [16]:
# Frontera eficiente: tres activos con inversión a partes iguales

resultados = []

for trio in combinations(retornos.columns, 3):
    t = list(trio)
    pesos = np.array([1/3, 1/3, 1/3])

    rent = np.dot(r_anual[t], pesos)
    cov = cov_anual.loc[t, t].values
    vol = np.sqrt(pesos @ cov @ pesos)

    resultados.append({
        "Trio": " - ".join(t),
        "Rent": rent,
        "Vol": vol,
        "Sharpe": rent / vol,
        "Pesos": pesos
    })

df_carteras_iguales = pd.DataFrame(resultados)

# Cartera óptima
mejor_igual = df_carteras_iguales.loc[
    df_carteras_iguales["Sharpe"].idxmax()
]

# Frontera eficiente
orden = df_carteras_iguales.sort_values(
    ["Vol", "Rent"], ascending=[True, False]
)

df_frontera_igual = orden[
    orden["Rent"] > orden["Rent"].cummax().shift(fill_value=-np.inf) + 1e-8
]

# Etiqueta de la cartera óptima
etiqueta = (
    f"<b>CARTERA ÓPTIMA</b><br>{mejor_igual['Trio']}<br>"
    f"Rentabilidad: {mejor_igual['Rent']:.2%}<br>"
    f"Volatilidad: {mejor_igual['Vol']:.2%}<br>"
    f"Sharpe: {mejor_igual['Sharpe']:.3f}<br>"
    f"Pesos: 33,33 % cada activo"
)

# Gráfico
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_carteras_iguales["Vol"], y=df_carteras_iguales["Rent"],
    mode="markers", name="Carteras",
    marker=dict(color=paleta_bbva["azul_celeste_aqua"], size=6, opacity=0.5)
))

fig.add_trace(go.Scatter(
    x=df_frontera_igual["Vol"], y=df_frontera_igual["Rent"],
    mode="lines+markers", name="Frontera eficiente",
    line=dict(color=paleta_bbva["azul_logotipo"], width=2.5)
))

fig.add_trace(go.Scatter(
    x=[mejor_igual["Vol"]], y=[mejor_igual["Rent"]],
    mode="markers", name="Cartera óptima",
    marker=dict(color=paleta_bbva["azul_oscuro_digital"], size=17, symbol="star")
))

fig.add_annotation(
    x=mejor_igual["Vol"], y=mejor_igual["Rent"], text=etiqueta,
    showarrow=True, ax=100, ay=-90,
    bgcolor="white", bordercolor=paleta_bbva["azul_logotipo"],
    borderpad=7, font=dict(size=11)
)

fig.update_layout(
    title="Frontera Eficiente - Tres Activos con Pesos Iguales",
    xaxis_title="Volatilidad Anualizada",
    yaxis_title="Rentabilidad Anualizada",
    xaxis_tickformat=".0%", yaxis_tickformat=".0%",
    width=1100, height=750
)

fig.write_image(os.path.join(ruta_graficos, ('Frontera Eficiente - Tres Activos con Pesos Iguales' + '.png').replace(' ', '_')))

fig.show()

# Resultados
print("CARTERA ÓPTIMA")
print("-----------------------------")
print("Activos:", mejor_igual["Trio"])
print(f"Rentabilidad: {mejor_igual['Rent']:.2%}")
print(f"Volatilidad: {mejor_igual['Vol']:.2%}")
print(f"Sharpe: {mejor_igual['Sharpe']:.3f}")
print(f"\nTotal de carteras analizadas: {len(df_carteras_iguales):,}")

CARTERA ÓPTIMA
-----------------------------
Activos: MSFT - GS - AAPL
Rentabilidad: 27.30%
Volatilidad: 24.72%
Sharpe: 1.104

Total de carteras analizadas: 2,600


In [11]:
# Selección de carteras con criterios de correlación, riesgo bajista y COVID

rf_diario = 0.03 / 252

# Correlaciones móviles de 80 días
rolling_corr = retornos.rolling(80).corr().dropna()
corr_media = rolling_corr.groupby(level=1).mean()

fechas = rolling_corr.index.get_level_values(0).unique()
fechas_covid = fechas[
    (fechas >= "2020-02-19") & (fechas <= "2020-11-09")
]

candidatas = []

for _, c in df_frontera_igual.iterrows():
    t = c["Trio"].split(" - ")

    # Ordenar los pares según su correlación media
    pares = sorted([
        (corr_media.loc[a, b], a, b)
        for a, b in combinations(t, 2)
    ])

    # Correlación media de los dos pares menos correlacionados durante COVID
    medias_covid = [
        rolling_corr.xs(a, level=1)[b].loc["2020-02-19":"2020-11-09"].mean()
        for _, a, b in pares[:2]
    ]

    # Rendimiento diario de la cartera a partes iguales
    rent_diaria = retornos[t].mean(axis=1)

    # Riesgo bajista y ratio de Sortino
    downside = np.minimum(0, rent_diaria - rf_diario)
    vol_negativa = np.sqrt(np.mean(downside**2)) * np.sqrt(252)

    sortino = (c["Rent"] - 0.03) / vol_negativa

    candidatas.append({
        "Trio": c["Trio"],
        "Rent": c["Rent"],
        "Vol_Negativa": vol_negativa,
        "Sortino": sortino,
        "Score_Crisis": np.mean(medias_covid),
        "Corr_Alta": pares[2][0],
        "Corr_Baja1": pares[0][0],
        "Corr_Baja2": pares[1][0]
    })

df_cand = pd.DataFrame(candidatas)

# Aplicar los criterios de correlación
df_cand = df_cand[
    (df_cand["Corr_Alta"] >= 0.45) &
    (df_cand["Corr_Baja1"] < 0.35) &
    (df_cand["Corr_Baja2"] < 0.35) &
    (df_cand["Score_Crisis"].notna())
].copy()

# Agrupar por rentabilidad y seleccionar el mejor Sortino
df_cand = df_cand.sort_values("Vol_Negativa")

df_cand["Grupo"] = pd.qcut(
    df_cand["Rent"],
    q=min(8, df_cand["Rent"].nunique()),
    duplicates="drop"
)

frontera_final = (
    df_cand.loc[
        df_cand.groupby("Grupo", observed=True)["Sortino"].idxmax()
    ]
    .sort_values("Score_Crisis")
    .reset_index(drop=True)
)

# Mostrar resultados en texto plano
print(frontera_final.to_string(index=False))

             Trio     Rent  Vol_Negativa  Sortino  Score_Crisis  Corr_Alta  Corr_Baja1  Corr_Baja2          Grupo
      GS - PG - V 0.187264      0.136206 1.154606      0.131212   0.451444    0.110642    0.272981 (0.183, 0.212]
MSFT - AAPL - CAT 0.266507      0.165358 1.430271      0.143927   0.632510    0.216836    0.240384 (0.257, 0.267]
    GS - PG - MCD 0.173207      0.136677 1.047772      0.198946   0.455529    0.110642    0.238366  (0.17, 0.183]
  MSFT - GS - CAT 0.253546      0.165508 1.350663      0.224172   0.594257    0.216836    0.306785 (0.241, 0.257]
    KO - JNJ - GS 0.150918      0.132083 0.915470      0.236181   0.456656    0.184902    0.213274   (0.15, 0.17]
 MSFT - GS - AAPL 0.273030      0.169753 1.431670      0.322518   0.632510    0.306785    0.328826 (0.267, 0.273]
 MSFT - KO - AAPL 0.231205      0.156956 1.281923      0.404518   0.632510    0.245554    0.270982 (0.212, 0.241]


Se selecciona la cartera compuesta por MSFT, GS y CAT